# 04 · Pesos de clase, SMOTENC o ninguna corrección — Morbilidad materna severa MIMIC-IV (M2)

| Campo | Contenido |
|---|---|
| Secuencia de notebooks | 01: E1 · E2 · E3 · E4 · P1 → 02: E5 · P2 → 03: E6 · E7 · P3 · R1 · E8 → **04: E8** → 05: E9 (E7) → 06: E9 (E3) → 07: E9 · C1-C3 · R2 · R4 → 08: E9 · C4 → 09: E9 · C4 → 10: P4 · E10 · E11 |
| Qué hace | Compara tres formas de manejar el desbalance de clases con el procedimiento anidado y agrupado por paciente del notebook 03: ninguna corrección, pesos de clase y sobremuestreo SMOTENC dentro de cada ajuste. Cada estrategia se ajusta por separado y se compara después (y antes) de la recalibración de Platt |
| Etapas y puertas | E8 (decisión de modelado sobre el desbalance) · E9 |
| Entradas | Base local PostgreSQL de MIMIC-IV (tablas de `sql/`) (mismas tablas que el notebook 03) |
| Salidas | `data/outputs_resampling.txt` |
| Uso de los datos | Solo se muestran y guardan resultados agregados; nada a nivel de paciente. No agregar celdas que muestren o exporten filas |
| Tiempo | 30–60 minutos (tres corridas anidadas y el bootstrap) |
| Siguiente | `05_creatinine_indicator_sensitivity_smm_es.ipynb` |
| Versión en inglés | `04_resampling_comparison_smm.ipynb` (mismo código) |

| Estrategia | Pesos de clase | Remuestreo |
|---|---|---|
| `unweighted` | ninguno (`class_weight=None`, `scale_pos_weight=1`) | ninguno |
| `weighted` | balanceados / `sqrt(neg/pos)` en XGBoost | ninguno |
| `smote` | **ninguno** (pesos y sobremuestreo nunca se combinan) | SMOTENC en la parte de entrenamiento de cada ajuste; razón minoritaria/mayoritaria ajustada (0,1 o 1,0) |

> **Orden de ejecución.** Correr todas las celdas de arriba hacia abajo. Primero `QUICK = True` como prueba rápida (minutos; nunca se reportan sus cifras) y después `QUICK = False`.

## E1 · Gobierno y configuración — librerías, conexión y configuración

Fija la semilla, los pliegues externos e internos, las repeticiones, las réplicas del bootstrap y los conteos de referencia `EXPECTED` congelados en la puerta P1. `QUICK = True` es una prueba rápida (minutos); sus cifras nunca se reportan.

In [1]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False        # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

# ----------------------------------------------------------------------------- language and presentation
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
# Nothing below changes a computation: it only decides how tables and messages are shown. The English aggregate
# reports data/outputs_*.txt are written exactly as before (the article cites them); the Spanish notebook also
# writes a translated copy, data/outputs_*_es.txt.
LANG = "es"
import builtins, contextlib, importlib, io, html as _html
import numpy as np
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)
def _tr(s, lang=None):
    """Translate one line keeping its indentation and trailing spaces."""
    lang = lang or LANG
    if lang != "es" or not isinstance(s, str):
        return s
    out = []
    for ln in s.split("\n"):
        core = ln.strip()
        if not core:
            out.append(ln); continue
        i = ln.index(core)
        out.append(ln[:i] + translate(core, "es") + ln[i + len(core):])
    return "\n".join(out)

_REC = None          # while emit() runs: the blocks of the English report, [("text", line) | ("table", frame, index)]

def print(*args, sep=" ", end="\n", file=None, flush=False):
    """Inside emit() the English text goes to the report unchanged; outside it, messages follow LANG."""
    if _REC is not None and file is None:
        _REC.append(("text", sep.join(str(a) for a in args)))
        return builtins.print(*args, sep=sep, end=end, flush=flush)
    if LANG == "es":
        args = [_tr(a, "es") for a in args]
    return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)

def _cell_fmt(v):
    # Display format of a cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1000:
            return f"{int(v):,}"
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _html_table(df, index):
    df = df.astype(object)
    df = df.map(_cell_fmt) if hasattr(df, "map") else df.applymap(_cell_fmt)
    return f"<div style='max-height:420px; overflow:auto'>{df.to_html(index=index, na_rep='', border=0)}</div>"

def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

def table(df, index=False, title=None):
    """Show a table. Inside emit() it also prints the English text of the report (df.to_string)."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
        index = True
    if _REC is not None:
        _REC.append(("table", df, index))
        return builtins.print(df.to_string(index=index))
    if not _in_notebook():
        return builtins.print(translate(df, LANG).to_string(index=index))
    from IPython.display import HTML, display as _display
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>" if title else ""
    _display(HTML(head + _html_table(translate(df, LANG), index)))

def _blocks_html(blocks):
    """Report blocks as HTML: a text line right before a table is its title; rules of '=' are dropped."""
    out, pending = [], []
    def flush_text():
        for ln in pending:
            if ln.strip():
                out.append(f"<div style='white-space:pre-wrap; font-family:monospace; margin:2px 0'>{_html.escape(_tr(ln.rstrip()))}</div>")
        pending.clear()
    for b in blocks:
        if b[0] == "text":
            pending.extend(l for l in b[1].split("\n") if not set(l.strip()) <= {"="} or not l.strip())
        else:
            title = None
            while pending and not pending[-1].strip():
                pending.pop()
            if pending:
                title = pending.pop().strip()
            flush_text()
            if title:
                out.append(f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>")
            out.append(_html_table(translate(b[1], LANG), b[2]))
    flush_text()
    return "\n".join(out)

def _blocks_text_es(blocks):
    """Spanish copy of a report (data/outputs_*_es.txt)."""
    lines = []
    for b in blocks:
        if b[0] == "text":
            lines.extend(_tr(l, "es") for l in b[1].split("\n"))
        else:
            lines.extend(translate(b[1], "es").to_string(index=b[2]).split("\n"))
    return "\n".join(lines) + "\n"

def _emit_to(path, fn, show=True):
    """Run fn, write its English text to `path` exactly as printed and, from the Spanish notebook, a translated copy
    (*_es.txt); show the report as tables."""
    global _REC
    buf = io.StringIO()
    _REC = []
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    finally:
        blocks, _REC = _REC, None
    text = buf.getvalue()
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    if show:
        show_report(blocks)
    if LANG == "es":
        with open(path.replace(".txt", "_es.txt"), "w", encoding="utf-8") as fh:
            fh.write(_blocks_text_es(blocks))
    return text

def show_report(blocks):
    if _in_notebook():
        from IPython.display import HTML, display as _display
        _display(HTML(_blocks_html(blocks)))
    else:
        for b in blocks:
            builtins.print(_tr(b[1]) if b[0] == "text" else translate(b[1], LANG).to_string(index=b[2]))

def notebook_summary(rows, title="Notebook summary"):
    """Summary of the notebook (as in the neonatal repository): one value per row; a value that cannot be computed
    (for example, an analysis that was not run in this session) is shown as n/a."""
    out = {}
    for label, fn in rows:
        try:
            v = fn() if callable(fn) else fn
        except Exception:
            v = "n/a"
        out[label] = v
    table(pd.Series(out, name="value").to_frame(), index=True, title=title)

def _m(res, model, met, d=3):
    """Point value of a metric with its 95% bootstrap interval, from a result with "point" and "boot"."""
    v = f"{res['point'][(model, met)]:.{d}f}"
    b = res.get("boot") or {}
    a = b.get((model, met)) if isinstance(b, dict) else None
    if a is not None and len(a) >= 20:
        lo, hi = np.nanpercentile(a, [2.5, 97.5])
        v += f" ({lo:.{d}f}-{hi:.{d}f})"
    return v

def _cv_auroc(res, model):
    """AUROC averaged over the repeats of a cross-validation result that only keeps out-of-fold predictions."""
    from sklearn.metrics import roc_auc_score
    y = res["y"]
    return float(np.mean([roc_auc_score(y, res["preds"][model][r]) for r in range(len(res["preds"][model]))]))


QUICK=False  outer=5x3  inner=3  B=1000  trees=300


## E7 · Predictores y E4 / P1 — tabla de predictores, diccionario final y conteos de referencia

`predictors_24h_v2` tiene una fila por hospitalización obstétrica con los predictores observados en la ventana (admisión − 24 h, admisión + 24 h). `predictor_dictionary_final_v2` asigna a cada variable su rol (`main model`, `ICU sensitivity only`, `laboratory sensitivity ...`) y sus límites de plausibilidad; los valores fuera de los límites pasan a faltantes. Los datos cargados se comparan con los conteos de referencia; la celda siguiente muestra la decisión de la puerta P1.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Se reproducen los tres conteos | P1 pasa: sigue a los análisis | 19.383 partos, 445 SMM, 3 muertes |
| Algún conteo difiere | Se detiene: revisar las tablas de `sql/` antes de ajustar cualquier modelo | — |

In [2]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

Roles del diccionario -> {'main': ['hemoglobin', 'creatinine'], 'ICU only': 14, 'lab sensitivity': 11, 'missingness indicator in main model': ['creatinine']}
Valores llevados a faltante por los límites de plausibilidad (conteo por variable): {'hemoglobin': 0, 'hematocrit': 0, 'creatinine': 1}
partos elegibles: 19383  (esperado 19383)  [OK]
SMM: 445  (esperado 445)  [OK]
muertes: 3  (esperado 3)  [OK]
Admisiones obstétricas elegibles de cualquier tipo: 23625 | SMM: 814 | deaths: 5


In [3]:
# Gate G1 decision: the loaded data must reproduce the reference counts frozen at the count audit (aggregates only)
g1 = pd.DataFrame([{"count": k, "loaded": v, "expected": EXPECTED[k],
                    "result": "reproduced" if v == EXPECTED[k] else "DIFFERS"} for k, v in chk.items()])
table(g1, title="Gate G1: reference counts")
table(pd.Series({"G1 decision": "passes: continue to the analyses" if (g1["result"] == "reproduced").all()
                 else "does NOT pass: review the tables of sql/ before fitting any model"}, name="decision"),
      title="Gate G1 decision")

conteo,cargado,esperado,resultado
partos elegibles,"19,383","19,383",reproducido
SMM,445,445,reproducido
muertes,3,3,reproducido


,decisión
decisión P1,pasa: se sigue con los análisis


## E6 · Punto de predicción y ventana — conjuntos de análisis y de predictores

Cada análisis es una especificación (población, conjunto de predictores, etiqueta). El punto de predicción es la admisión + 24 h; solo se usa la información registrada hasta ese momento.

| Nombre | Población | Predictores |
|---|---|---|
| `main` | Hospitalizaciones de parto elegibles | Variables del modelo principal del diccionario (hemoglobina, creatinina), indicador de creatinina faltante, edad, variables categóricas de la admisión |
| `first_admission` | Primera hospitalización de parto de cada paciente, si es elegible | como `main` |
| `any_obstetric` | Hospitalizaciones obstétricas elegibles (con o sin parto) | como `main` |
| `labs_extended` | como `main` | `main` + hematocrito y los demás laboratorios, cada uno con su indicador de faltante |
| `icu_vars` | como `main` | `main` + las variables solo de UCI (cada una con indicador de faltante) + `icu_in_24h` |
| `label_ge2` | como `main` | como `main`, desenlace = dos o más indicadores de SMM (`smm_ge2`) |
| `no_icu_24h` | Partos elegibles **sin** ingreso a UCI en las primeras 24 h | como `main` |
| `icd10_only` | Partos elegibles cuyos diagnósticos son todos CIE-10-CM (sin diagnósticos CIE-9-CM) | como `main` |

In [4]:
def make_spec(name):
    d = df[df.eligible_24h].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    label = "y"
    if name in ("main", "labs_extended", "icu_vars", "label_ge2", "no_icu_24h", "icd10_only"):
        d = d[d.is_delivery_v2]
        if name == "no_icu_24h":
            d = d[~d.icu_in_24h]
    elif name == "first_admission":
        first = df[df.is_delivery_v2].sort_values(["subject_id", "admittime", "hadm_id"]).drop_duplicates("subject_id")
        d = d[d.hadm_id.isin(first.hadm_id)]
    elif name == "any_obstetric":
        pass
    else:
        raise ValueError(name)
    if name == "icd10_only":
        ver = pd.read_sql("SELECT hadm_id, has_icd9_dx, has_icd10_dx FROM mimiciv.cohort_obstetric_v2", engine)
        ver["has_icd9_dx"] = ver["has_icd9_dx"].fillna(False).astype(bool)
        ver["has_icd10_dx"] = ver["has_icd10_dx"].fillna(False).astype(bool)
        keep = ver.loc[ver.has_icd10_dx & ~ver.has_icd9_dx, "hadm_id"]
        d = d[d.hadm_id.isin(keep)]
    if name == "labs_extended":
        num = num + VAR_LAB
        binc = binc + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" not in binc]
    if name == "icu_vars":
        num = num + VAR_ICU
        binc = binc + [f"{v}_missing" for v in VAR_ICU] + ["icu_in_24h_bin"]
    if name == "label_ge2":
        g2 = pd.read_sql("SELECT hadm_id, smm_ge2 FROM mimiciv.cohort_obstetric_v2", engine)
        g2["smm_ge2"] = g2["smm_ge2"].fillna(False).astype(bool).astype(int)
        d = d.merge(g2, on="hadm_id", how="left")
        d["y_ge2"] = d["smm_ge2"].fillna(0).astype(int)
        label = "y_ge2"
    # drop columns that do not exist or never vary (a constant indicator carries no information)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    cat  = [c for c in CAT_COLS if c in d.columns]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label=label)

for nm in ["main", "first_admission", "any_obstetric", "labs_extended", "icu_vars", "no_icu_24h", "icd10_only"]:
    sp = make_spec(nm)
    d = sp["data"]
    print(f"{nm:16s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

main: admisiones=19383, pacientes=13710, eventos=445, numéricas=3, binarias=1, categóricas=5
first_admission: admisiones=13701, pacientes=13701, eventos=356, numéricas=3, binarias=1, categóricas=5
any_obstetric: admisiones=23625, pacientes=14602, eventos=814, numéricas=3, binarias=1, categóricas=5
labs_extended: admisiones=19383, pacientes=13710, eventos=445, numéricas=14, binarias=12, categóricas=5
icu_vars: admisiones=19383, pacientes=13710, eventos=445, numéricas=17, binarias=16, categóricas=5
no_icu_24h: admisiones=19268, pacientes=13632, eventos=355, numéricas=3, binarias=1, categóricas=5
icd10_only: admisiones=8864, pacientes=7070, eventos=271, numéricas=3, binarias=1, categóricas=5


## E8 · Modelado, validación y métricas — preprocesamiento, algoritmos, ajuste anidado, ensamble y calibración

Todo lo que aprende de los datos queda dentro de la parte de entrenamiento de cada partición (puerta P2: ninguna paciente a ambos lados):

* `Winsor` recorta cada columna numérica a los percentiles 0,5–99,5 *de los datos de entrenamiento*.
* La imputación por la mediana, el escalado y la codificación one-hot se ajustan dentro de un `Pipeline`, una vez por ajuste.
* Los hiperparámetros se eligen con **validación cruzada interna agrupada por paciente** (precisión promedio de las predicciones fuera de muestra).
* El meta-modelo del ensamble se entrena con las predicciones **fuera de muestra** de los algoritmos base (nunca con predicciones dentro de muestra).
* El calibrador de Platt se entrena con predicciones fuera de muestra de las pacientes de entrenamiento; las pacientes de prueba externa solo se evalúan.

In [5]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID_ACTIVE)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID_ACTIVE[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    for n_ in names:
        out[f"{n_}_raw"] = pred_te[n_]      # base-learner probabilities before recalibration
    return out, chosen

def run_cv(spec, label=translate("", LANG)):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    globals()["N_NUM_ACTIVE"] = len(spec["num"])
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS + RAW_KEYS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in preds:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

## E8 · Modelado, validación y métricas — métricas, bootstrap por conglomerados y diferencias pareadas

La discriminación, la calibración y la utilidad clínica se calculan con las predicciones fuera de muestra de cada repetición y se promedian entre repeticiones. El bootstrap remuestrea **pacientes** (todas las admisiones de una paciente se mueven juntas), así que los intervalos reflejan las admisiones repetidas. Las diferencias entre modelos usan las **mismas** muestras bootstrap (pareadas).

In [6]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

## E8 · E11 Reporte — función de reporte

Arma el reporte agregado de un análisis: discriminación, calibración, diferencias pareadas entre modelos, eventos captados en el 5% / 10% de mayor riesgo, beneficio neto, inestabilidad entre repeticiones y los hiperparámetros elegidos en los ciclos internos. `emit()` lo guarda como `data/outputs_<análisis>.txt` (en inglés, como lo cita el artículo) y lo muestra como tablas; el notebook en español también escribe `data/outputs_<análisis>_es.txt`.

In [7]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label=translate("ideal", LANG))
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label=translate("Logistic regression", LANG))
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label=translate("Stacked ensemble", LANG))
    ax.set_xlabel(translate("Mean predicted risk (decile)", LANG)); ax.set_ylabel(translate("Observed SMM rate", LANG))
    ax.set_title(translate(f"Calibration by decile -- {res['spec']['name']}", LANG)); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Write the aggregate report data/outputs_<name>.txt (English text, as cited by the article) and show it as tables;
    the Spanish notebook also writes data/outputs_<name>_es.txt. Aggregates only."""
    path = f"../data/outputs_{name}.txt"
    text = _emit_to(path, fn)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name == "main":
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    table(sl[sl.repeat == 1].drop(columns=["repeat"]))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    table(pd.DataFrame(rows))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    table(pd.DataFrame(rows))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    table(pd.DataFrame(rows))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    table(pd.DataFrame(rows))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        table(pd.DataFrame(rows))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    table(decile_table(res))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    table(tab)

## E8 · Estrategias de desbalance — pesos, SMOTENC y ninguna corrección

`SmoteNCAuto` sobremuestrea la clase minoritaria **dentro del pipeline**, después del preprocesamiento ajustado solo con entrenamiento, así que las pacientes sintéticas se crean solo con filas de entrenamiento del ajuste en curso. Las columnas one-hot y los indicadores se tratan como categóricos en SMOTENC (los fija el voto de los vecinos en lugar de promediarlos).

In [8]:
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

RAW_KEYS = [f"{n}_raw" for n in ["LR", "RF", "ET", "XGB"]]
N_NUM_ACTIVE = 3
BASES = ["LR", "RF", "ET", "XGB"]
STRATS = ["unweighted", "weighted", "smote"]
LABEL = {"unweighted": "No weights, no resampling", "weighted": "Class weights only", "smote": "SMOTENC only (no weights)"}

class SmoteNCAuto(BaseEstimator):
    """SMOTENC on the columns after the numeric block (one-hot and indicator columns are categorical)."""
    def __init__(self, n_num=3, ratio=1.0, seed=0):
        self.n_num = n_num; self.ratio = ratio; self.seed = seed
    def fit_resample(self, X, y):
        X = np.asarray(X)
        cats = list(range(self.n_num, X.shape[1]))
        if cats:
            smp = SMOTENC(categorical_features=cats, sampling_strategy=self.ratio, k_neighbors=5, random_state=self.seed)
        else:
            smp = SMOTE(sampling_strategy=self.ratio, k_neighbors=5, random_state=self.seed)
        return smp.fit_resample(X, y)

def make_grid(strategy):
    lr_c, rf_d, et_d, xgb_d = (0.01, 0.1, 1.0), (2, 4, 8), (4, 8, 12), (1, 2, 4)
    if strategy == "main":
        return GRID
    if strategy in ("unweighted", "smote"):
        g = {"LR": [dict(C=c, cw=None) for c in lr_c], "RF": [dict(depth=d, cw=None) for d in rf_d],
             "ET": [dict(depth=d, cw=None) for d in et_d], "XGB": [dict(depth=d, spw=1) for d in xgb_d]}
        if strategy == "smote":
            g = {k: [dict(c, ratio=r) for c in v for r in (0.1, 1.0)] for k, v in g.items()}
        return g
    if strategy == "weighted":
        return {"LR": [dict(C=c, cw="balanced") for c in lr_c], "RF": [dict(depth=d, cw="balanced_subsample") for d in rf_d],
                "ET": [dict(depth=d, cw="balanced_subsample") for d in et_d], "XGB": [dict(depth=d, spw="sqrt") for d in xgb_d]}
    raise ValueError(strategy)

def set_strategy(strategy):
    global GRID_ACTIVE
    GRID_ACTIVE = make_grid(strategy)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    clf = make_clf(name, cfg, ytr, seed)
    if "ratio" in cfg:
        pipe = ImbPipeline([("prep", clone(prep)), ("smote", SmoteNCAuto(N_NUM_ACTIVE, cfg["ratio"], seed)), ("clf", clf)])
    else:
        pipe = Pipeline([("prep", clone(prep)), ("clf", clf)])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

set_strategy("main")
print({s: sum(len(v) for v in make_grid(s).values()) for s in STRATS}, "configurations per strategy")

{'unweighted': 12, 'weighted': 12, 'smote': 24} configurations per strategy


## E8 · E9 — correr las tres estrategias (anidado, agrupado por paciente)

In [9]:
res_strat = {}
for st in STRATS:
    print(f"=== strategy: {st} ({LABEL[st]})")
    set_strategy(st)
    res_strat[st] = run_cv(make_spec("main"))
set_strategy("main")

=== estrategia: sin pesos (Sin pesos ni remuestreo)
[main] el primer pliegue externo tardó 0.4 min -> total proyectado de unos 6 min
[main] validación cruzada terminada en 5.4 min; cada admisión evaluada exactamente una vez por repetición: Sí
=== estrategia: con pesos (Solo pesos de clase)
[main] el primer pliegue externo tardó 0.5 min -> total proyectado de unos 7 min
[main] validación cruzada terminada en 7.4 min; cada admisión evaluada exactamente una vez por repetición: Sí
=== estrategia: SMOTENC (Solo SMOTENC (sin pesos))
[main] el primer pliegue externo tardó 4.4 min -> total proyectado de unos 67 min
[main] validación cruzada terminada en 81.0 min; cada admisión evaluada exactamente una vez por repetición: Sí


## E8 · E11 — comparación, diferencias pareadas y reporte

Las filas calibradas usan el paso de Platt del pipeline; las filas *antes de la recalibración* usan las probabilidades tal como las produjo cada algoritmo. Las diferencias comparten las mismas muestras bootstrap de pacientes (pareadas).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| El remuestreo no mejora la discriminación y distorsiona las probabilidades originales | Sin remuestreo en el framework; los pesos de clase siguen como opción de la malla | SMOTENC bajó el AUROC (XGBoost 0,775 frente a 0,789) y, como los pesos de clase, distorsionó los riesgos originales (O/E 0,27–0,29) |
| El remuestreo mejora la discriminación después de recalibrar | Agregarlo como opción de la malla | — |

In [10]:
def light(y, p, w):
    z = logit(p)
    return {"auroc": roc_auc_score(y, p, sample_weight=w), "auprc": average_precision_score(y, p, sample_weight=w),
            "brier": float(np.average((p - y) ** 2, weights=w)), "oe": float(np.sum(w * y) / np.sum(w * p)),
            "meanp": float(np.sum(w * p) / w.sum()), "slope": calib_slope(y, z, w)}

def strategy_compare(res_strat, B, seed):
    first = res_strat[STRATS[0]]
    y = first["y"].astype(float); g = first["g"]
    P = {}
    for st in STRATS:
        for m in BASES + ["Stack"]:
            P[(m, st)] = res_strat[st]["preds"][m]
        for m in BASES:
            P[(m + "_raw", st)] = res_strat[st]["preds"][m + "_raw"]
    def all_light(w):
        out = {}
        for key, arr in P.items():
            acc = [light(y, arr[r], w) for r in range(N_REPEATS)]
            for k in acc[0]:
                out[(key, k)] = float(np.mean([a[k] for a in acc]))
        return out
    point = all_light(np.ones(len(y)))
    ug, inv = np.unique(g, return_inverse=True)
    rng = np.random.default_rng(seed)
    boot = {}
    t0 = time.time()
    for b in range(B):
        w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
        cur = all_light(w)
        for k, v in cur.items():
            boot.setdefault(k, []).append(v)
        for m in BASES + ["Stack"]:
            for a_, b_ in [("smote", "unweighted"), ("weighted", "unweighted"), ("smote", "weighted")]:
                for met in ("auroc", "auprc", "brier"):
                    boot.setdefault(((m, f"{a_}-{b_}"), met), []).append(cur[((m, a_), met)] - cur[((m, b_), met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return point, {k: np.array(v) for k, v in boot.items()}, y

def strategy_report(res_strat, point, boot, y):
    ci = lambda key, d=3: (lambda lo, hi: f" ({lo:.{d}f}-{hi:.{d}f})")(*np.percentile(boot[key], [2.5, 97.5]))
    print("=" * 110)
    print(f"RESAMPLING COMPARISON: SMM, eligible deliveries, n={len(y)}, events={int(y.sum())}, prevalence={y.mean():.4f}; "
          f"outer {OUTER_FOLDS}x{N_REPEATS}, inner {INNER_FOLDS}, bootstrap B={len(next(iter(boot.values())))}, QUICK={QUICK}")
    print("=" * 110)
    rows = []
    for m in BASES + ["Stack"]:
        for st in STRATS:
            k = (m, st)
            rows.append({"model": m, "strategy": LABEL[st],
                         "auroc": f"{point[(k,'auroc')]:.3f}" + ci((k, "auroc")), "auprc": f"{point[(k,'auprc')]:.3f}" + ci((k, "auprc")),
                         "brier": f"{point[(k,'brier')]:.5f}" + ci((k, "brier"), 5),
                         "slope": f"{point[(k,'slope')]:.2f}" + ci((k, "slope"), 2), "O/E": f"{point[(k,'oe')]:.2f}"})
    print("\n[S1] After recalibration (Platt): discrimination and calibration; 95% cluster-bootstrap intervals")
    table(pd.DataFrame(rows))

    rows = []
    for m in BASES:
        for st in STRATS:
            k = (m + "_raw", st)
            rows.append({"base learner": m, "strategy": LABEL[st], "mean predicted risk": f"{point[(k,'meanp')]:.4f}",
                         "observed/expected": f"{point[(k,'oe')]:.2f}", "calibration slope": f"{point[(k,'slope')]:.2f}",
                         "auroc": f"{point[(k,'auroc')]:.3f}"})
    print(f"\n[S2] Before recalibration (raw probabilities of each algorithm); observed prevalence = {y.mean():.4f}")
    table(pd.DataFrame(rows))

    rows = []
    for m in BASES + ["Stack"]:
        for a_, b_ in [("weighted", "unweighted"), ("smote", "unweighted"), ("smote", "weighted")]:
            k = (m, f"{a_}-{b_}")
            r = {"model": m, "comparison (first minus second)": f"{a_} - {b_}"}
            for met, d in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
                r[met] = f"{point[((m, a_), met)] - point[((m, b_), met)]:+.{d}f}" + ci((k, met), d)
            rows.append(r)
    print("\n[S3] Paired differences between strategies (same bootstrap samples)")
    table(pd.DataFrame(rows))

    print("\n[S4] Configurations selected in the inner loops (per learner, out of %d outer fits)" % (N_REPEATS * OUTER_FOLDS))
    for st in STRATS:
        ch = pd.DataFrame(res_strat[st]["chosen"], columns=["learner", "config"])
        tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
        print(f"  -- {LABEL[st]}")
        table(tab)

point_s, boot_s, y_s = strategy_compare(res_strat, BOOT_SENS, SEED + 11)
emit("resampling", lambda: strategy_report(res_strat, point_s, boot_s, y_s))

  bootstrap: 5 réplicas tardaron 8 s -> proyectado 8.3 min para B=300


modelo,estrategia,AUROC,AUPRC,Brier,pendiente,O/E
LR,Sin pesos ni remuestreo,0.780 (0.754-0.802),0.155 (0.123-0.191),0.02086 (0.01891-0.02280),1.00 (0.91-1.08),1.00
LR,Solo pesos de clase,0.781 (0.755-0.803),0.149 (0.118-0.182),0.02095 (0.01901-0.02290),0.97 (0.88-1.04),0.97
LR,Solo SMOTENC (sin pesos),0.757 (0.731-0.781),0.142 (0.114-0.174),0.02105 (0.01911-0.02305),1.00 (0.90-1.08),1.00
RF,Sin pesos ni remuestreo,0.777 (0.751-0.800),0.158 (0.126-0.190),0.02075 (0.01884-0.02266),1.01 (0.93-1.08),1.01
RF,Solo pesos de clase,0.780 (0.757-0.801),0.136 (0.113-0.166),0.02110 (0.01928-0.02297),1.00 (0.92-1.08),0.96
RF,Solo SMOTENC (sin pesos),0.773 (0.747-0.797),0.135 (0.110-0.165),0.02106 (0.01927-0.02295),1.02 (0.93-1.09),0.99
ET,Sin pesos ni remuestreo,0.770 (0.745-0.793),0.151 (0.120-0.180),0.02090 (0.01900-0.02276),1.00 (0.92-1.07),1.00
ET,Solo pesos de clase,0.768 (0.744-0.792),0.129 (0.107-0.153),0.02119 (0.01929-0.02307),1.01 (0.92-1.08),0.99
ET,Solo SMOTENC (sin pesos),0.759 (0.734-0.785),0.152 (0.120-0.182),0.02095 (0.01910-0.02289),1.01 (0.92-1.08),0.99
XGB,Sin pesos ni remuestreo,0.789 (0.764-0.811),0.165 (0.133-0.199),0.02074 (0.01884-0.02266),1.02 (0.94-1.10),1.00


[reporte agregado guardado como data/outputs_resampling.txt  (7295 caracteres)]


## Resumen del notebook

Discriminación y calibración con las tres estrategias para el desbalance de clases.

In [11]:
# Summary of notebook 04
rows_s = []
for _mod in ("XGB", "Stack"):
    for _st in STRATS:
        rows_s.append((f"AUROC {_mod}, {LABEL[_st]}", lambda m=_mod, s=_st: f"{point_s[((m, s), 'auroc')]:.3f}"))
rows_s.append(("observed/expected of the raw risks, XGBoost with SMOTENC", lambda: f"{point_s[(('XGB_raw', 'smote'), 'oe')]:.2f}"))
rows_s.append(("observed/expected of the raw risks, XGBoost without resampling",
               lambda: f"{point_s[(('XGB_raw', 'unweighted'), 'oe')]:.2f}"))
notebook_summary(rows_s)

,valor
"AUROC XGB, Sin pesos ni remuestreo",0.789
"AUROC XGB, Solo pesos de clase",0.792
"AUROC XGB, Solo SMOTENC (sin pesos)",0.775
"AUROC Stack, Sin pesos ni remuestreo",0.786
"AUROC Stack, Solo pesos de clase",0.790
"AUROC Stack, Solo SMOTENC (sin pesos)",0.774
"observados/esperados de los riesgos crudos, XGBoost con SMOTENC",0.29
"observados/esperados de los riesgos crudos, XGBoost sin remuestreo",1.01
